# Plot Kimodo Constraints As Skeletons

Notebook para inspecionar cada constraint do jeito que ela entra no Kimodo.

Para `fullbody`, o notebook desenha tr\u00eas painéis por keyframe:

- **Col A — pose3d original**: esqueleto do sujeito original reconstru\u00eddo a partir de `source_pose3d_npz_path`, reordenado para SMPLX22, grounded e rebased do mesmo jeito que o anchor. Reflete as propor\u00e7\u00f5es ósseas do sujeito (priors do pipeline).
- **Col B — FK do anchor retargetado**: posi\u00e7\u00f5es globais reconstru\u00eddas a partir do `local_joints_rot` + `root_positions` do constraint, usando a rest-pose SMPLX22 do Kimodo via FK. Reflete as propor\u00e7\u00f5es ósseas do Kimodo — é a cinemática que o modelo efetivamente consome.
- **Col C — round-trip `FullBodyConstraintSet.from_dict`**: o mesmo da Col B, mas obtido via `constraint_set.global_joints_positions`. Serve para verificar numericamente que a parse pelo Kimodo é idêntica ao FK direto.

A diferen\u00e7a de propor\u00e7\u00f5es entre Col A e Col B é esperada por design; o que precisa bater frame-a-frame é a **dire\u00e7\u00e3o dos ossos**.

In [1]:
%pip install matplotlib

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import json
import math
import sys

import matplotlib.pyplot as plt
import numpy as np

plt.style.use("seaborn-v0_8-whitegrid")
np.set_printoptions(suppress=True, precision=4)


In [ ]:
PROMPT_ENTRY_PATH = Path(
    "output/robot_emotions_kimodo_generated_single_cc/robot_emotions_10ms_u02_tag11__w000/prompt_entry.json"
)
CONSTRAINTS_PATH = None

MAX_SKELETON_COLUMNS = 4
SKELETON_PANEL_SIZE = 3.8
CENTER_EACH_FRAME_ON_ROOT = False

# Se quiser abrir um constraints.json diretamente, deixe PROMPT_ENTRY_PATH = None
# e ajuste CONSTRAINTS_PATH para o arquivo desejado.


In [4]:
def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "kimodo" / "kimodo").exists() and (candidate / "robot_emotions_vlm").exists():
            return candidate
    raise FileNotFoundError("Nao consegui localizar a raiz do repo a partir do cwd atual.")


def import_local_kimodo(repo_root: Path):
    kimodo_repo = repo_root / "kimodo"
    kimodo_repo_str = str(kimodo_repo)
    if kimodo_repo_str not in sys.path:
        sys.path.insert(0, kimodo_repo_str)

    import kimodo  # noqa: F401
    from kimodo.constraints import FullBodyConstraintSet, Root2DConstraintSet
    from kimodo.skeleton import build_skeleton

    return FullBodyConstraintSet, Root2DConstraintSet, build_skeleton


def load_json(path: Path):
    path = Path(path)
    return json.loads(path.read_text(encoding="utf-8"))


def resolve_constraint_bundle(prompt_entry_path=None, constraints_path=None):
    prompt_entry = None
    resolved_constraints_path = None

    if prompt_entry_path is not None:
        prompt_entry = load_json(Path(prompt_entry_path))
        resolved_constraints_path = Path(prompt_entry["constraints_path"])

    if constraints_path is not None:
        resolved_constraints_path = Path(constraints_path)

    if resolved_constraints_path is None:
        raise ValueError("Defina PROMPT_ENTRY_PATH ou CONSTRAINTS_PATH.")

    constraints = load_json(resolved_constraints_path)
    traceability_path = resolved_constraints_path.with_name("traceability.json")
    traceability = load_json(traceability_path) if traceability_path.exists() else None

    return {
        "prompt_entry": prompt_entry,
        "constraints_path": resolved_constraints_path,
        "constraints": constraints,
        "traceability_path": traceability_path if traceability_path.exists() else None,
        "traceability": traceability,
    }


def summarize_bundle(bundle):
    prompt_entry = bundle["prompt_entry"]
    traceability = bundle["traceability"]

    print(f"constraints_path: {bundle['constraints_path']}")
    if bundle["traceability_path"] is not None:
        print(f"traceability_path: {bundle['traceability_path']}")

    if prompt_entry is not None:
        print(f"prompt_id: {prompt_entry['prompt_id']}")
        print(f"reference_clip_id: {prompt_entry['reference_clip_id']}")
        print(f"prompt_text: {prompt_entry['prompt_text']}")

    if traceability is not None:
        print(f"constraint_mode: {traceability['constraint_mode']}")
        print(f"target_fps: {traceability['target_fps']}")
        print(f"constraint_types: {traceability['constraint_types']}")
        print(f"ground_height_m: {traceability['ground_height_m']:.4f}")


REPO_ROOT = find_repo_root()
FullBodyConstraintSet, Root2DConstraintSet, build_skeleton = import_local_kimodo(REPO_ROOT)


/home/henriquesouza/miniconda3/envs/kimodo/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
def build_smplx22_skeleton():
    return build_skeleton(22)


def source_keyframe_labels(traceability, count: int):
    if traceability is None:
        return [None] * count
    source_keyframes = traceability.get("source_keyframes")
    if source_keyframes is None or len(source_keyframes) != count:
        return [None] * count
    return [int(value) for value in source_keyframes]


def compute_plot_positions(global_joints_positions: np.ndarray, center_on_root: bool) -> np.ndarray:
    positions = np.asarray(global_joints_positions, dtype=np.float32).copy()
    if center_on_root:
        positions -= positions[:, [0], :]
    return positions


def compute_axis_limits(plot_positions: np.ndarray):
    xyz = plot_positions.reshape(-1, 3)
    mins = xyz.min(axis=0)
    maxs = xyz.max(axis=0)
    center = 0.5 * (mins + maxs)
    radius = 0.5 * float(np.max(maxs - mins))
    radius = max(radius, 0.5)
    return center, radius * 1.1


def plot_single_skeleton(ax, joints_xyz, joint_parents, title: str, center, radius):
    joints_xyz = np.asarray(joints_xyz, dtype=np.float32)
    for joint_index, parent_index in enumerate(joint_parents):
        if int(parent_index) < 0:
            continue
        xs = [joints_xyz[int(parent_index), 0], joints_xyz[joint_index, 0]]
        ys = [joints_xyz[int(parent_index), 2], joints_xyz[joint_index, 2]]
        zs = [joints_xyz[int(parent_index), 1], joints_xyz[joint_index, 1]]
        ax.plot(xs, ys, zs, color="tab:blue", linewidth=2)

    ax.scatter(joints_xyz[:, 0], joints_xyz[:, 2], joints_xyz[:, 1], color="black", s=18)
    ax.scatter(joints_xyz[0, 0], joints_xyz[0, 2], joints_xyz[0, 1], color="tab:red", s=40)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("x")
    ax.set_ylabel("z")
    ax.set_zlabel("y")
    ax.view_init(elev=18, azim=-72)
    ax.set_xlim(center[0] - radius, center[0] + radius)
    ax.set_ylim(center[2] - radius, center[2] + radius)
    ax.set_zlim(center[1] - radius, center[1] + radius)


def plot_fullbody_constraint(constraint, traceability=None, *, constraint_index=0, max_cols=4, panel_size=3.8, center_on_root=False):
    skeleton = build_smplx22_skeleton()
    constraint_set = FullBodyConstraintSet.from_dict(skeleton, constraint)

    frame_indices = np.asarray(constraint["frame_indices"], dtype=np.int32)
    root_positions = np.asarray(constraint["root_positions"], dtype=np.float32)
    smooth_root_2d = np.asarray(constraint.get("smooth_root_2d", root_positions[:, [0, 2]]), dtype=np.float32)
    global_joints_positions = constraint_set.global_joints_positions.detach().cpu().numpy()
    plot_positions = compute_plot_positions(global_joints_positions, center_on_root=center_on_root)
    joint_parents = constraint_set.skeleton.joint_parents.detach().cpu().numpy()
    joint_names = list(constraint_set.skeleton.bone_order_names)
    source_labels = source_keyframe_labels(traceability, len(frame_indices))
    center, radius = compute_axis_limits(plot_positions)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4), constrained_layout=True)
    axes[0].plot(frame_indices, root_positions[:, 0], marker="o", label="x")
    axes[0].plot(frame_indices, root_positions[:, 1], marker="o", label="y")
    axes[0].plot(frame_indices, root_positions[:, 2], marker="o", label="z")
    axes[0].set_title(f"fullbody[{constraint_index}] root_positions")
    axes[0].set_xlabel("target frame")
    axes[0].set_ylabel("meters")
    axes[0].legend()

    axes[1].plot(smooth_root_2d[:, 0], smooth_root_2d[:, 1], marker="o")
    axes[1].scatter(smooth_root_2d[0, 0], smooth_root_2d[0, 1], label="start", s=60)
    axes[1].scatter(smooth_root_2d[-1, 0], smooth_root_2d[-1, 1], label="end", s=60)
    axes[1].set_title(f"fullbody[{constraint_index}] root trajectory (x,z)")
    axes[1].set_xlabel("x")
    axes[1].set_ylabel("z")
    axes[1].axis("equal")
    axes[1].legend()
    plt.show()

    n_frames = len(frame_indices)
    n_cols = min(max_cols, n_frames)
    n_rows = math.ceil(n_frames / n_cols)
    fig = plt.figure(figsize=(panel_size * n_cols, panel_size * n_rows), constrained_layout=True)
    note = "centered on root" if center_on_root else "global positions"
    fig.suptitle(f"fullbody[{constraint_index}] skeleton per keyframe ({note})", fontsize=14)

    for subplot_index in range(n_rows * n_cols):
        ax = fig.add_subplot(n_rows, n_cols, subplot_index + 1, projection="3d")
        if subplot_index >= n_frames:
            ax.set_axis_off()
            continue

        title = f"target {int(frame_indices[subplot_index])}"
        source_label = source_labels[subplot_index]
        if source_label is not None:
            title += f"\nsource {source_label}"

        plot_single_skeleton(
            ax,
            plot_positions[subplot_index],
            joint_parents,
            title,
            center,
            radius,
        )

    plt.show()

    print("joint_names:")
    print(joint_names)


def plot_root2d_constraint(constraint, *, constraint_index=0):
    frames = np.asarray(constraint["frame_indices"], dtype=np.int32)
    root2d = np.asarray(constraint["smooth_root_2d"], dtype=np.float32)
    heading = None if "global_root_heading" not in constraint else np.asarray(constraint["global_root_heading"], dtype=np.float32)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4), constrained_layout=True)
    axes[0].plot(frames, root2d[:, 0], label="x")
    axes[0].plot(frames, root2d[:, 1], label="z")
    axes[0].set_title(f"root2d[{constraint_index}] coordinates")
    axes[0].set_xlabel("target frame")
    axes[0].set_ylabel("meters")
    axes[0].legend()

    axes[1].plot(root2d[:, 0], root2d[:, 1], marker="o")
    axes[1].scatter(root2d[0, 0], root2d[0, 1], label="start", s=60)
    axes[1].scatter(root2d[-1, 0], root2d[-1, 1], label="end", s=60)
    if heading is not None:
        step = max(1, len(root2d) // 20)
        axes[1].quiver(
            root2d[::step, 0],
            root2d[::step, 1],
            heading[::step, 0],
            heading[::step, 1],
            angles="xy",
            scale_units="xy",
            scale=1.0,
            width=0.004,
            color="tab:red",
        )
    axes[1].set_title(f"root2d[{constraint_index}] trajectory")
    axes[1].set_xlabel("x")
    axes[1].set_ylabel("z")
    axes[1].axis("equal")
    axes[1].legend()
    plt.show()


In [6]:
def _load_source_pose3d_keyframes(traceability, *, to_kimodo_space=False):
    """Rebuild SMPLX22 keyframe positions from the reference pose3d.npz exactly as the anchor did."""
    if traceability is None:
        raise ValueError("traceability.json is required to rebuild the original pose3d panel.")

    from pose_module.interfaces import PoseSequence3D
    from robot_emotions_vlm.anchor_catalog import (
        _estimate_ground_height,
        _rebase_window_to_first_root_xz,
        _resolve_smplx_source_indices,
    )

    source_path = Path(traceability["source_pose3d_npz_path"])
    with np.load(source_path, allow_pickle=False) as payload:
        sequence = PoseSequence3D.from_npz_payload({key: payload[key] for key in payload.files})

    source_frame_ids = np.asarray(sequence.frame_indices, dtype=np.int32)
    source_frame_to_index = {int(frame_id): index for index, frame_id in enumerate(source_frame_ids.tolist())}
    missing = [int(frame_id) for frame_id in traceability["source_keyframes"] if int(frame_id) not in source_frame_to_index]
    if missing:
        raise ValueError(f"traceability source_keyframes not found in pose3d.npz: {missing}")
    keyframe_indices = np.asarray([source_frame_to_index[int(frame_id)] for frame_id in traceability["source_keyframes"]], dtype=np.int32)

    window_start = int(traceability["source_frame_start"])
    window_end = int(traceability["source_frame_end"])
    window_start_idx = source_frame_to_index[window_start]
    window_end_idx = source_frame_to_index[window_end] + 1

    smplx_source_indices = _resolve_smplx_source_indices(sequence.joint_names_3d)
    window_positions = np.asarray(
        sequence.joint_positions_xyz[window_start_idx:window_end_idx][:, smplx_source_indices, :],
        dtype=np.float32,
    )
    root_translation_m = sequence.resolved_root_translation_m()
    if root_translation_m is None:
        raise ValueError("Reference pose3d.npz is missing root_translation_m.")
    window_root = np.asarray(root_translation_m[window_start_idx:window_end_idx], dtype=np.float32)

    ground_height_m = float(_estimate_ground_height(window_positions))
    window_positions_grounded = window_positions.copy()
    window_positions_grounded[:, :, 1] -= np.float32(ground_height_m)
    window_root_grounded = window_root.copy()
    window_root_grounded[:, 1] -= np.float32(ground_height_m)
    window_positions_grounded[:, 0, :] = window_root_grounded

    dense_root_2d_placeholder = window_root_grounded[:, [0, 2]].astype(np.float32, copy=False)
    (window_positions_rebased, _, _, _) = _rebase_window_to_first_root_xz(
        window_positions_grounded,
        window_root_grounded,
        dense_root_2d_placeholder,
    )

    keyframes_relative = keyframe_indices - np.int32(window_start_idx)
    sparse_positions = window_positions_rebased[keyframes_relative].astype(np.float32, copy=False)
    if to_kimodo_space:
        sparse_positions = sparse_positions.copy()
        sparse_positions[:, :, 0] *= -1
        sparse_positions[:, :, 2] *= -1
    return sparse_positions, ground_height_m


def _anchor_positions_via_fk(constraint):
    """Apply axis_angle + FK on the Kimodo skeleton, matching what FullBodyConstraintSet.from_dict does internally."""
    import torch
    from kimodo.geometry import axis_angle_to_matrix

    skeleton = build_smplx22_skeleton()
    local_rot = torch.tensor(np.asarray(constraint["local_joints_rot"], dtype=np.float32))
    root_positions = torch.tensor(np.asarray(constraint["root_positions"], dtype=np.float32))
    local_rot_mats = axis_angle_to_matrix(local_rot)
    _, global_joints_positions, _ = skeleton.fk(local_rot_mats, root_positions)
    return global_joints_positions.detach().cpu().numpy().astype(np.float32, copy=False)


def _constraint_authoritative_positions(constraint, anchor_fk_positions):
    """Return the fullbody geometry Kimodo will effectively constrain."""
    if "global_joints_positions" in constraint:
        return (
            np.asarray(constraint["global_joints_positions"], dtype=np.float32),
            "constraint global_joints_positions (authoritative)",
        )
    return anchor_fk_positions, "constraint local_joints_rot + root_positions (FK)"


def plot_fullbody_tri_panel(
    constraint,
    traceability,
    *,
    constraint_index=0,
    max_cols=MAX_SKELETON_COLUMNS,
    panel_size=SKELETON_PANEL_SIZE,
):
    if "local_joints_rot" not in constraint:
        print(f"fullbody[{constraint_index}] missing local_joints_rot; skipping tri-panel comparison.")
        return

    skeleton = build_smplx22_skeleton()
    joint_parents = skeleton.joint_parents.detach().cpu().numpy()

    source_positions, ground_height_m = _load_source_pose3d_keyframes(
        traceability,
        to_kimodo_space=True,
    )
    anchor_fk_positions = _anchor_positions_via_fk(constraint)
    authoritative_positions, authoritative_label = _constraint_authoritative_positions(
        constraint,
        anchor_fk_positions,
    )
    constraint_set = FullBodyConstraintSet.from_dict(skeleton, constraint)
    kimodo_internal_positions = constraint_set.global_joints_positions.detach().cpu().numpy().astype(np.float32, copy=False)

    max_internal_vs_authoritative = float(np.max(np.abs(kimodo_internal_positions - authoritative_positions)))
    if "global_joints_positions" in constraint:
        max_local_fk_vs_authoritative = float(np.max(np.abs(anchor_fk_positions - authoritative_positions)))
        print(
            f"fullbody[{constraint_index}] ground_height_m={ground_height_m:.4f}  "
            f"max |local FK - authoritative| = {max_local_fk_vs_authoritative:.3e} m  "
            f"max |Kimodo internal - authoritative| = {max_internal_vs_authoritative:.3e} m"
        )
    else:
        print(
            f"fullbody[{constraint_index}] ground_height_m={ground_height_m:.4f}  "
            f"round-trip max |FK - Kimodo| = {max_internal_vs_authoritative:.3e} m"
        )

    all_positions = np.concatenate([source_positions, authoritative_positions, kimodo_internal_positions], axis=0)
    center, radius = compute_axis_limits(all_positions)

    frame_indices = np.asarray(constraint["frame_indices"], dtype=np.int32)
    source_labels = source_keyframe_labels(traceability, len(frame_indices))
    n_frames = len(frame_indices)

    fig = plt.figure(figsize=(panel_size * 3, panel_size * n_frames), constrained_layout=True)
    fig.suptitle(
        f"fullbody[{constraint_index}] tri-panel: source(Kimodo space)  |  constraint authoritative  |  Kimodo internal",
        fontsize=14,
    )
    columns = [
        ("pose3d source projected to Kimodo space", source_positions),
        (authoritative_label, authoritative_positions),
        ("Kimodo internal after load", kimodo_internal_positions),
    ]
    for row_index in range(n_frames):
        target_label = int(frame_indices[row_index])
        source_label = source_labels[row_index]
        for col_index, (title_suffix, positions) in enumerate(columns):
            subplot_index = row_index * 3 + col_index + 1
            ax = fig.add_subplot(n_frames, 3, subplot_index, projection="3d")
            title = f"target {target_label}  |  {title_suffix}"
            if source_label is not None:
                title = f"source {source_label}\n{title}"
            plot_single_skeleton(
                ax,
                positions[row_index],
                joint_parents,
                title,
                center,
                radius,
            )
    plt.show()

In [7]:
bundle = resolve_constraint_bundle(
    prompt_entry_path=PROMPT_ENTRY_PATH,
    constraints_path=CONSTRAINTS_PATH,
)
summarize_bundle(bundle)

for index, constraint in enumerate(bundle["constraints"]):
    frame_count = len(constraint["frame_indices"])
    print(f"\nconstraint[{index}] type={constraint['type']} frames={frame_count}")
    if constraint["type"] == "fullbody":
        plot_fullbody_constraint(
            constraint,
            bundle["traceability"],
            constraint_index=index,
            max_cols=MAX_SKELETON_COLUMNS,
            panel_size=SKELETON_PANEL_SIZE,
            center_on_root=CENTER_EACH_FRAME_ON_ROOT,
        )
        plot_fullbody_tri_panel(
            constraint,
            bundle["traceability"],
            constraint_index=index,
            max_cols=MAX_SKELETON_COLUMNS,
            panel_size=SKELETON_PANEL_SIZE,
        )
    elif constraint["type"] == "root2d":
        plot_root2d_constraint(constraint, constraint_index=index)
    else:
        print("Tipo ainda nao tratado neste notebook.")

constraints_path: /home/henriquesouza/IMUGPT/output/robot_emotions_kimodo_anchors_hands/robot_emotions_10ms_u02_tag11__w000/constraints.json
traceability_path: /home/henriquesouza/IMUGPT/output/robot_emotions_kimodo_anchors_hands/robot_emotions_10ms_u02_tag11__w000/traceability.json
prompt_id: robot_emotions_10ms_u02_tag11__w000
reference_clip_id: robot_emotions_10ms_u02_tag11
prompt_text: A person stands still, hands clasped together, head slightly tilted, arms moving subtly, legs stationary
constraint_mode: pose3d
target_fps: 30.0
constraint_types: ['left-hand', 'right-hand', 'root2d']


KeyError: 'ground_height_m'